# ML-04 — Search Intelligence Data Contract

This notebook defines the data contract for the Search Intelligence recovery-forecasting task.

**Decision-time rule:** features must use information that would be knowable at the decision date. The future 30-day outcome is reserved for the label.

**Verification principle:** every material contract claim is backed by a query and an observed result.


## 1. Unit of analysis + time window

### Contract statement

**Unit of analysis:** one `(content_hash_id, report_date)` observation.

**Time window verified here:** March 1–31, 2026.

The March slice contains one observation per content/date combination, with no duplicate grain groups.


### 1.1 Verify grain

We group by `content_hash_id` and `report_date` and check whether any combination appears more than once.


In [2]:
con.sql(f"""
    WITH grain_check AS (
        SELECT content_hash_id, report_date, COUNT(*) AS n
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
        GROUP BY content_hash_id, report_date
    )
    SELECT
        COUNT(*) AS unique_grain_combos,
        SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS duplicate_grain_groups,
        MAX(n) AS max_rows_in_one_group
    FROM grain_check
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬────────────────────────┬───────────────────────┐
│ unique_grain_combos │ duplicate_grain_groups │ max_rows_in_one_group │
│        int64        │         int128         │         int64         │
├─────────────────────┼────────────────────────┼───────────────────────┤
│             9841378 │                      0 │                     1 │
└─────────────────────┴────────────────────────┴───────────────────────┘

**Observed:** 9,841,378 unique grain combinations, **0 duplicate groups**, and a maximum group size of **1**.

**Conclusion:** the claimed `(content_hash_id, report_date)` grain holds for the March slice.


### 1.2 Verify row count and date coverage

This checks the total number of rows, the first and last dates, and how many distinct report dates are present.


In [3]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_dates_present
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬────────────────────────┐
│ total_rows │  min_date  │  max_date  │ distinct_dates_present │
│   int64    │    date    │    date    │         int64          │
├────────────┼────────────┼────────────┼────────────────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │                     31 │
└────────────┴────────────┴────────────┴────────────────────────┘

**Observed:** 9,841,378 rows from 2026-03-01 through 2026-03-31, with **31 distinct dates**.

**Conclusion:** all 31 March dates are represented in the inspected slice.


## 2. Fields: feature / label / context / excluded

The contract separates fields by their role at decision time.

### Label

**`gsc_impressions`** is the source metric for the recovery label. It is used to construct the historical baseline and future outcome, so it is **not itself a model feature**.

### Core features

The selected five feature candidates are:

| Field | Role | Why it is useful |
|---|---|---|
| `gsc_clicks` | Feature | Captures current search-response activity and is knowable from the current/baseline period; it is correlated with the label's underlying visibility mechanics but does not contain the future 30-day window. |
| `gsc_avg_position` | Feature | Captures current search ranking/visibility quality; available when `gsc_data_available` is true, with a documented zero-value data-quality caveat. |
| `ga4_engaged_sessions` | Feature | Captures the breadth of meaningful user engagement among sessions. |
| `ga4_total_engagement_sec` | Feature | Captures engagement depth/total engaged time, providing a different behavioral dimension from engaged-session count. |
| `sessions_organic` | Feature | Captures organic-session volume from the analytics side, complementing GSC visibility/click signals with observed traffic behavior. |

**Coverage caveat:** the three GA4/session features are only directly observed on the `ga4_data_available = TRUE` subset, which is sparse and changes across March.

### Gates / eligibility conditions

- `gsc_data_available`: **gate**, not a model feature. Rows are eligible for GSC-based signals only when this flag is `TRUE`.
- `ga4_data_available`: **gate**, not a model feature. GA4-derived features are usable only when this flag is `TRUE`.

### Context

- `report_date`, `month`: temporal context used for slicing/splitting and interpretation, not direct predictive features.
- `client_hash_id`, `content_hash_id`: structural identifiers used to define the observation and preserve page/client grouping.

### Excluded / deferred

- `gsc_sum_position`: not selected; the contract uses `gsc_avg_position` as the ranking signal.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`: different traffic phenomenon from the GSC-impressions recovery target; deferred rather than forced into the core five.
- `scroll_events`: potentially meaningful behavioral signal, but deferred because only **4.21%** of March rows have `ga4_data_available = TRUE`, making its usable coverage too sparse for the core five.
- `search_volume`: not present in the contracted fact table schema; excluded unless a specific, verified join source is introduced.
- Other unselected `sessions_*` fields: not selected for the core five. `sessions_direct`, `sessions_referral`, `sessions_social`, and `sessions_ai` were not individually gate-tested in this notebook and should be treated as assumed-by-extension, not proven.


## 3. Verify it with queries

Each query below is grouped by the contract claim it supports. The goal is not to collect queries for their own sake; each one answers a specific data-quality or availability question.


### 3.1 Schema inventory

**Query 1 — What columns actually exist in the contracted fact table?**


In [11]:
schema_df = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    LIMIT 1
""").df()

for col in schema_df['column_name']:
    print(col)

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


**Interpretation:** The full schema is printed so field bucketing is based on the actual contracted table, not on assumptions from another dataset.


### 3.2 GSC availability: does the flag vary within clients?

**Query 2 — Is `gsc_data_available` actually row/day-varying?**


In [5]:
con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(DISTINCT gsc_data_available) AS distinct_values
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY client_hash_id
    HAVING COUNT(DISTINCT gsc_data_available) > 1
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬─────────────────┐
│     client_hash_id      │ distinct_values │
│         varchar         │      int64      │
├─────────────────────────┼─────────────────┤
│ client_62f4a7e64f5e0096 │               2 │
│ client_9958f0a7ae1df715 │               2 │
│ client_73cda7b4e4f265ea │               2 │
│ client_fef1a8f436438636 │               2 │
│ client_08a6a72ff48e62c0 │               2 │
│ client_ba65e80a1116ae41 │               2 │
│ client_3ffa76342f366962 │               2 │
│ client_f623b01661d4bfe4 │               2 │
│ client_cd12bcfd98942aa1 │               2 │
│ client_a80fca3f171ed1de │               2 │
│            ·            │               · │
│            ·            │               · │
│            ·            │               · │
│ client_0fa64a184f18a4a0 │               2 │
│ client_86ebc2f12c01f586 │               2 │
│ client_0797ff3a1fc9a6a5 │               2 │
│ client_59256b0571e0c970 │               2 │
│ client_7eafe750768f0ac2 │       

**Interpretation:** 47 clients show more than one distinct availability state in March. This supports treating the flag as a row-level availability signal rather than a static client attribute.


### 3.3 GSC availability: NULL vs populated values

**Query 3 — Does availability correspond to real vs placeholder impressions?**


In [6]:
con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS row_count,
        COUNT(gsc_impressions) AS non_null_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY gsc_data_available
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬──────────────────────┐
│ gsc_data_available │ row_count │ non_null_impressions │
│      boolean       │   int64   │        int64         │
├────────────────────┼───────────┼──────────────────────┤
│ false              │   6230317 │              6230317 │
│ true               │   3611061 │              3611061 │
└────────────────────┴───────────┴──────────────────────┘

**Interpretation:** Both groups are non-null, so NULL-ness alone does not establish availability. This motivates the next zero-inflation check.


### 3.4 GSC availability: zero-imputation check

**Query 4 — Are unavailable GSC impressions zero-imputed?**


In [7]:
con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬──────────────────┬─────────────────┐
│ gsc_data_available │ row_count │ zero_impressions │ avg_impressions │
│      boolean       │   int64   │      int128      │     double      │
├────────────────────┼───────────┼──────────────────┼─────────────────┤
│ false              │   6230317 │          6230317 │             0.0 │
│ true               │   3611061 │                0 │           77.72 │
└────────────────────┴───────────┴──────────────────┴─────────────────┘

**Interpretation:** FALSE rows are 100% zero with average 0.0, while TRUE rows contain non-zero impressions with average 77.72. This is strong evidence that `gsc_data_available` gates the usable GSC signal.


### 3.5 GSC client-level cross-check

**Query 5 — Is `gsc_data_available` merely inherited from `client_has_gsc`?**


In [8]:
con.sql(f"""
    SELECT
        client_has_gsc,
        gsc_data_available,
        COUNT(*) AS row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY client_has_gsc, gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────────────┬───────────┐
│ client_has_gsc │ gsc_data_available │ row_count │
│    boolean     │      boolean       │   int64   │
├────────────────┼────────────────────┼───────────┤
│ true           │ false              │   6230317 │
│ true           │ true               │   3611061 │
└────────────────┴────────────────────┴───────────┘

**Interpretation:** All March rows have `client_has_gsc = TRUE`, while `gsc_data_available` splits into TRUE/FALSE. This supports the interpretation that availability is not simply the client-level integration flag.


### 3.6 GSC survivor count

**Query 6 — How many March rows survive the required GSC availability filter?**


In [10]:
con.sql(f"""
    SELECT COUNT(*) AS rows_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ rows_available │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

**Interpretation:** Filtering with `gsc_data_available IS TRUE` leaves 3,611,061 rows. This is the auditable survivor count used by the contract.


### 3.7 GA4/scroll availability behavior

**Query 7 — Does `ga4_data_available` govern `scroll_events`?**


In [12]:
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN scroll_events = 0 THEN 1 ELSE 0 END) AS zero_scroll_events,
        ROUND(AVG(scroll_events), 2) AS avg_scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬────────────────────┬───────────────────┐
│ ga4_data_available │ row_count │ zero_scroll_events │ avg_scroll_events │
│      boolean       │   int64   │       int128       │      double       │
├────────────────────┼───────────┼────────────────────┼───────────────────┤
│ NULL               │   3018741 │                  0 │              NULL │
│ false              │   6408671 │            6408671 │               0.0 │
│ true               │    413966 │             290926 │              0.53 │
└────────────────────┴───────────┴────────────────────┴───────────────────┘

**Interpretation:** FALSE rows are entirely zero, NULL rows have NULL scroll values, while TRUE rows contain observed non-zero values but also legitimate zeros. This supports using `ga4_data_available` as the gate while recognizing that zero is a real outcome for this metric.


### 3.8 GA4 coverage

**Query 8 — What fraction of March has GA4 data available?**


In [13]:
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_march
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬──────────────┐
│ ga4_data_available │ row_count │ pct_of_march │
│      boolean       │   int64   │    double    │
├────────────────────┼───────────┼──────────────┤
│ false              │   6408671 │        65.12 │
│ NULL               │   3018741 │        30.67 │
│ true               │    413966 │         4.21 │
└────────────────────┴───────────┴──────────────┘

**Interpretation:** March coverage is 4.21% TRUE, 65.12% FALSE, and 30.67% NULL. This establishes the sparse-coverage caveat for GA4-derived features.


### 3.9 GA4 feature-block behavior

**Query 9 — Do GA4/session candidates share the same gate?**


In [14]:
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count,
        ROUND(AVG(ga4_pageviews), 2) AS avg_pageviews,
        ROUND(AVG(ga4_sessions), 2) AS avg_ga4_sessions,
        ROUND(AVG(ga4_users), 2) AS avg_users,
        ROUND(AVG(ga4_engaged_sessions), 2) AS avg_engaged_sessions,
        ROUND(AVG(ga4_total_engagement_sec), 2) AS avg_engagement_sec,
        ROUND(AVG(sessions_organic), 2) AS avg_sessions_organic,
        ROUND(AVG(sessions_paid), 2) AS avg_sessions_paid
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬───────────────┬──────────────────┬───────────┬──────────────────────┬────────────────────┬──────────────────────┬───────────────────┐
│ ga4_data_available │ row_count │ avg_pageviews │ avg_ga4_sessions │ avg_users │ avg_engaged_sessions │ avg_engagement_sec │ avg_sessions_organic │ avg_sessions_paid │
│      boolean       │   int64   │    double     │      double      │  double   │        double        │       double       │        double        │      double       │
├────────────────────┼───────────┼───────────────┼──────────────────┼───────────┼──────────────────────┼────────────────────┼──────────────────────┼───────────────────┤
│ NULL               │   3018741 │          NULL │             NULL │      NULL │                 NULL │               NULL │                 NULL │              NULL │
│ false              │   6408671 │           0.0 │              0.0 │       0.0 │                  0.0 │                0.0 │                  0.0 │       

**Interpretation:** The tested GA4 and session columns are NULL when the flag is NULL, zero when FALSE, and non-zero on TRUE rows. This confirms the gate for the tested block; four other `sessions_*` columns remain assumed-by-extension.


### 3.10 GA4 feature diversity

**Query 10 — Are engaged sessions and total engagement time redundant?**


In [15]:
con.sql(f"""
    SELECT
        CORR(ga4_engaged_sessions, ga4_total_engagement_sec) AS corr
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND ga4_data_available IS TRUE
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│        corr        │
│       double       │
├────────────────────┤
│ 0.6320979012436048 │
└────────────────────┘

**Interpretation:** Correlation is 0.632, so the two metrics are related but not highly redundant. Keeping both preserves breadth-vs-depth behavioral information.


### 3.11 GSC availability by day

**Query 11 — Is there an obvious month-end GSC availability dip?**


In [16]:
con.sql(f"""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS available_rows,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY report_date
    ORDER BY report_date
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬────────────────┬───────────────┐
│ report_date │ total_rows │ available_rows │ pct_available │
│    date     │   int64    │     int128     │    double     │
├─────────────┼────────────┼────────────────┼───────────────┤
│ 2026-03-01  │     275874 │         101910 │         36.94 │
│ 2026-03-02  │     276269 │         103696 │         37.53 │
│ 2026-03-03  │     311676 │         107362 │         34.45 │
│ 2026-03-04  │     311675 │         109377 │         35.09 │
│ 2026-03-05  │     311676 │         109740 │         35.21 │
│ 2026-03-06  │     312187 │         110037 │         35.25 │
│ 2026-03-07  │     312387 │         102153 │          32.7 │
│ 2026-03-08  │     313374 │         101170 │         32.28 │
│ 2026-03-09  │     313874 │         111313 │         35.46 │
│ 2026-03-10  │     314047 │         113051 │          36.0 │
│     ·       │        ·   │            ·   │            ·  │
│     ·       │        ·   │            ·   │            ·  │
│     · 

**Interpretation:** Availability fluctuates through March but does not show a clear month-end collapse. The evidence supports the careful claim 'no observed month-end availability decline' rather than claiming that no reporting lag exists.


### 3.12 GSC average-position quality

**Query 12 — Are there zero placeholders in `gsc_avg_position`?**


In [17]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS zero_position_rows,
        MIN(gsc_avg_position) AS min_position,
        MAX(gsc_avg_position) AS max_position
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬──────────────┬──────────────┐
│ total_rows │ zero_position_rows │ min_position │ max_position │
│   int64    │       int128       │    double    │    double    │
├────────────┼────────────────────┼──────────────┼──────────────┤
│    3611061 │             163189 │          0.0 │        498.0 │
└────────────┴────────────────────┴──────────────┴──────────────┘

**Interpretation:** Among 3,611,061 GSC-available rows, 163,189 have position 0 (~4.52%). Since 0 is not a valid SERP position, rolling/aggregate calculations must explicitly handle these values.


### 3.13 GA4 availability by day

**Query 13 — Is GA4 coverage stable across March?**


In [18]:
con.sql(f"""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY report_date
    ORDER BY report_date
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬────────────────┬───────────────┐
│ report_date │ total_rows │ available_rows │ pct_available │
│    date     │   int64    │     int128     │    double     │
├─────────────┼────────────┼────────────────┼───────────────┤
│ 2026-03-01  │     275874 │           4909 │          1.78 │
│ 2026-03-02  │     276269 │           5349 │          1.94 │
│ 2026-03-03  │     311676 │           6831 │          2.19 │
│ 2026-03-04  │     311675 │           6791 │          2.18 │
│ 2026-03-05  │     311676 │           7653 │          2.46 │
│ 2026-03-06  │     312187 │           8389 │          2.69 │
│ 2026-03-07  │     312387 │           7865 │          2.52 │
│ 2026-03-08  │     313374 │           7589 │          2.42 │
│ 2026-03-09  │     313874 │           9653 │          3.08 │
│ 2026-03-10  │     314047 │          11731 │          3.74 │
│     ·       │        ·   │            ·   │            ·  │
│     ·       │        ·   │            ·   │            ·  │
│     · 

**Interpretation:** Coverage rises from 1.78% on March 1 to roughly 5–6% later in the month. This is not a simple month-end lag dip; it indicates that GA4 instrumentation coverage is changing over time and should be treated as a non-stationarity limitation.


## 4. Data limits

The verification work surfaced several limitations that must remain explicit in the contract:

1. **GSC zero-imputation:** when `gsc_data_available = FALSE`, `gsc_impressions` is zero-imputed. If these zeros enter rolling baseline/forward averages, they can artificially depress the calculated signal.
2. **`gsc_avg_position` zero values:** 163,189 of 3,611,061 GSC-available rows (~4.52%) have position 0. Any rolling/aggregate use must explicitly exclude or handle these values.
3. **GA4 sparsity:** only 4.21% of March rows have `ga4_data_available = TRUE`; 65.12% are FALSE and 30.67% are NULL.
4. **GA4 coverage is non-stationary within March:** availability rises from ~1.8% early in the month to ~5–6% later, so GA4 feature reliability should not be assumed constant across time.
5. **Untested session columns:** `sessions_direct`, `sessions_referral`, `sessions_social`, and `sessions_ai` were not individually verified against `ga4_data_available`; treating them as GA4-gated is an assumption-by-extension.
6. **Temporal/window limits:** a 30-day historical baseline and 30-day forward outcome require sufficient observations on both sides of a decision date. Edge dates and incomplete histories cannot support a fully formed label without additional handling.


## 5. Leakage trap

**Status:** to be added after the leakage experiment is designed and run.

Planned trap: a centered rolling average of `gsc_impressions` can accidentally include future observations from the label's forward window. The experiment will be performed only after a suitable multi-month time range is loaded so that both 30-day historical and 30-day future windows are computable.


## Self-check

- [x] Unit of analysis is stated and verified.
- [x] Field roles are explicitly bucketed with reasons.
- [x] Each major availability/quality claim has an adjacent query and interpretation.
- [x] Observed results are distinguished from assumptions and limitations.
- [ ] Leakage trap experiment is added and verified.
- [ ] Run the notebook top-to-bottom before submission.
- [ ] Confirm no private client names, URLs, or private queries are included.
- [ ] Commit the final notebook under `work/notebooks/`.
